## Download Dataset from Roboflow Universe

In [1]:
import os
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY"))
project = rf.workspace("fyp-vfrgn").project("imbalanced-abgb0")
version = project.version(2)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...


### Verify dataset structure

In [7]:
# Cell 3: Verify dataset structure
import os
from pathlib import Path

dataset_dir = Path("imbalanced-2")

# Check data.yaml exists
yaml_path = dataset_dir / "data.yaml"
if yaml_path.exists():
    print(f"Found: {yaml_path}")
    with open(yaml_path) as f:
        print(f.read())
else:
    print("data.yaml not found! Check directory:")
    for p in sorted(dataset_dir.rglob("*.yaml")):
        print(f"  {p}")

# Check splits
for split in ["train", "valid", "test"]:
    img_dir = dataset_dir / split / "images"
    lbl_dir = dataset_dir / split / "labels"
    if img_dir.exists():
        n_imgs = len(list(img_dir.glob("*.*")))
        n_lbls = len(list(lbl_dir.glob("*.txt")))
        print(f"{split}: {n_imgs} images, {n_lbls} labels")

Found: imbalanced-2\data.yaml
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 7
names: ['Bus', 'Motorcycle', 'Pickup', 'Sedan', 'Suv', 'Truck', 'Van']

roboflow:
  workspace: fyp-vfrgn
  project: imbalanced-abgb0
  version: 2
  license: CC BY 4.0
  url: https://app.roboflow.com/fyp-vfrgn/imbalanced-abgb0/2
train: 5820 images, 5820 labels
valid: 728 images, 728 labels
test: 726 images, 726 labels


### Verify GPU

In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Device Count: {torch.cuda.device_count()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
CUDA Version: 12.1
Device Count: 1
GPU: NVIDIA GeForce RTX 4060
VRAM: 8.6 GB


### Fine-tune YOLO26s

In [9]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

results = model.train(
    data="imbalanced-2/data.yaml",

    # --- Core ---
    epochs=150,
    batch=16,
    imgsz=640,
    patience=20,

    # --- Optimizer ---
    optimizer="AdamW",
    lr0=1e-3,
    lrf=0.01,
    weight_decay=5e-4,
    warmup_epochs=3.0,

    # --- Loss weights ---
    cls=1.5,
    box=7.5,

    # --- Augmentation ---
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    close_mosaic=10,

    # --- Logging ---
    project="vehicle_detection",
    name="yolo26s_imbalanced",
    plots=True,
    save=True,
    val=True,
)

Ultralytics 8.4.24  Python-3.11.9 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=imbalanced-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26s_imbalanced, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, pati

### Evaluate Fine-tuned Model

Before benchmarking the model, we need to load the best saved checkpoint. To ensure it fits on the GPU, we first need to free up GPU memory. This involves deleting any remaining references to previously used objects, triggering Python’s garbage collector, and clearing the CUDA memory cache.

In [10]:
import gc
import torch
import weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):

    if not torch.cuda.is_available():
        if verbose:
            print("[INFO] CUDA is not available. No GPU cleanup needed.")
        return

    def get_memory_stats():
        allocated = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        return allocated, reserved

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

    # Ensure we drop all strong references
    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("[WARNING] Object not fully garbage collected yet.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

In [11]:
cleanup_gpu_memory(model, verbose=True)

[Before] Allocated: 256.08 MB | Reserved: 564.00 MB
[WARNING] Object not fully garbage collected yet.
[After]  Allocated: 256.08 MB | Reserved: 412.00 MB


In [3]:
from ultralytics import YOLO

model = YOLO("../runs/detect/vehicle_detection/yolo26s_imbalanced/weights/best.pt")
metrics = model.val(data="imbalanced-2/data.yaml", split="test")

print(f"\nOverall mAP@50: {metrics.box.map50:.4f}")
print(f"Overall mAP@50:95: {metrics.box.map:.4f}")

# Per-class metrics
for i, name in enumerate(metrics.names.values()):
    print(f"  {name}: mAP@50={metrics.box.ap50[i]:.4f}, "
          f"P={metrics.box.p[i]:.4f}, R={metrics.box.r[i]:.4f}")

Ultralytics 8.4.24  Python-3.11.9 torch-2.2.2+cu121 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
YOLO26s summary (fused): 122 layers, 9,467,889 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 118.430.5 MB/s, size: 55.9 KB)
val: Scanning C:\Users\user\Documents\Vehicle-Detection-at-R-R-\notebooks\imbalanced-2\test\labels.cache... 726 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 726/726  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 12, len(boxes) = 4140. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 46/46 5.7it/s 8.1s0.1s
                   all        726       4140        0.8      0.801      0.868      0.581
                   Bus          9          9      0.719      0.889      0.87

### Export for deployment (TensorRT)

In [4]:
from ultralytics import YOLO

model = YOLO("../runs/detect/vehicle_detection/yolo26s_imbalanced/weights/best.pt")
model.export(format="engine", half=True, imgsz=640)

WARNING TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.24  Python-3.11.9 torch-2.2.2+cu121 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
YOLO26s summary (fused): 122 layers, 9,467,889 parameters, 0 gradients, 20.5 GFLOPs

PyTorch: starting from '..\runs\detect\vehicle_detection\yolo26s_imbalanced\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (19.4 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB 1.3 MB/s eta 0:00:13
   ---------------------------------------- 0.1/16.4 MB 1.3 MB/s eta 0:00:13
   ---------------------------------------- 0.2/16.4 MB 1.5 MB/s eta 0:00:11
    --------------------------------------- 0.3/16.4 MB 1.9 MB/s eta 0:00:09
   - -------------------------------------- 0.5/16.4 MB 2.8 

Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.


ONNX: slimming with onnxslim 0.1.90...
ONNX: export success  41.0s, saved as '..\runs\detect\vehicle_detection\yolo26s_imbalanced\weights\best.onnx' (36.4 MB)

TensorRT: starting export with TensorRT 10.16.0.72...
TensorRT: input "images" with shape(1, 3, 640, 640) DataType.FLOAT
TensorRT: output "output0" with shape(1, 300, 6) DataType.FLOAT
TensorRT: building FP16 engine as ..\runs\detect\vehicle_detection\yolo26s_imbalanced\weights\best.engine
TensorRT: export success  265.4s, saved as '..\runs\detect\vehicle_detection\yolo26s_imbalanced\weights\best.engine' (21.4 MB)

Export complete (265.9s)
Results saved to C:\Users\user\Documents\Vehicle-Detection-at-R-R-\runs\detect\vehicle_detection\yolo26s_imbalanced\weights
Predict:         yolo predict task=detect model=..\runs\detect\vehicle_detection\yolo26s_imbalanced\weights\best.engine imgsz=640 half
Validate:        yolo val task=detect model=..\runs\detect\vehicle_detection\yolo26s_imbalanced\weights\best.engine imgsz=640 data=imbala

'..\\runs\\detect\\vehicle_detection\\yolo26s_imbalanced\\weights\\best.engine'